# Week 3 – Task 1: RAG over PDFs with ChromaDB

<small>Builds a RAG pipeline: discover PDFs → chunk → embed → store in ChromaDB → retrieve → generate answer via the reused Week 2 OpenRouter client. Tasks 2–5 are not implemented here.</small>


## Reusing the Week 2 Client

<small>Imports `ChatApp.ipynb` as a module via `import_ipynb`, so its `OpenRouterClient` isn't duplicated. Must run from the same folder as `ChatApp.ipynb`, `CONSTANTS.py`, and `.env`.</small>


In [117]:
# Reuse Week 2's OpenRouter client
import import_ipynb
import ChatApp

print("Reused from ChatApp.ipynb:")
print(" - OpenRouterClient  :", ChatApp.OpenRouterClient)
print(" - OpenRouterAPIError:", ChatApp.OpenRouterAPIError)
print(" - load_api_key()    :", ChatApp.load_api_key)
print(" - MODELS            :", ChatApp.MODELS)


Reused from ChatApp.ipynb:
 - OpenRouterClient  : <class 'ChatApp.OpenRouterClient'>
 - OpenRouterAPIError: <class 'ChatApp.OpenRouterAPIError'>
 - load_api_key()    : <function load_api_key at 0x000001E2903DECA0>
 - MODELS            : {'1': 'meta-llama/llama-3.1-8b-instruct', '2': 'google/gemma-3-27b-it', '3': 'nvidia/nemotron-nano-9b-v2:free'}


## Setup

<small>Extra packages needed: `pypdf`, `sentence-transformers`, `chromadb`, `import-ipynb`. `requests`/`python-dotenv` are reused from `ChatApp.ipynb`.</small>


In [118]:
# %pip install -q pypdf sentence-transformers chromadb import-ipynb

import os
import time
from pathlib import Path
from typing import List, Dict, Any, Optional

from pypdf import PdfReader
import chromadb
from sentence_transformers import SentenceTransformer


## The PDF Collection (`data/`)

<small>Four cheat-sheet PDFs (Python, ML, DL, AI) live in `data/`. No filenames are hardcoded — add/remove PDFs and re-run.</small>

| File | Topic | Author(s) |
|---|---|---|
| `python_cheatsheet.pdf` | Python | Eric Matthes |
| `machine_learning_cheatsheet.pdf` | ML (CS229) | Amidi & Amidi |
| `deep_learning_cheatsheet.pdf` | DL (CS230) | Amidi & Amidi |
| `artificial_intelligence_cheatsheet.pdf` | AI (CS221) | Amidi & Amidi |


## Configuration

<small>All pipeline settings live here — change and re-run from this point down.</small>


In [119]:
DATA_DIR = Path("data")
CHUNK_SIZE = 800
CHUNK_OVERLAP = 150
TOP_K = 4

EMBEDDING_MODEL_NAME = "all-MiniLM-L6-v2"

CHROMA_PERSIST_DIR = "chroma_store"
COLLECTION_NAME = "week3_pdf_knowledge_base"

GENERATION_MODEL_KEY = "1"
GENERATION_MODEL = ChatApp.MODELS[GENERATION_MODEL_KEY]

print(f"Chunking : {CHUNK_SIZE} chars, {CHUNK_OVERLAP} char overlap")
print(f"Retrieval: top-{TOP_K} chunks")
print(f"Embedding model: {EMBEDDING_MODEL_NAME}")
print(f"Generation model (via ChatApp): {GENERATION_MODEL}")


Chunking : 800 chars, 150 char overlap
Retrieval: top-4 chunks
Embedding model: all-MiniLM-L6-v2
Generation model (via ChatApp): meta-llama/llama-3.1-8b-instruct


## Step 1 — Discover PDFs

<small>Scans `data/` for `*.pdf` files at runtime. Raises `FileNotFoundError` if the folder is missing or empty.</small>


In [120]:
def discover_pdfs(data_dir: Path) -> List[Path]:
    """Return a sorted list of PDF paths found in data_dir."""
    if not data_dir.exists():
        raise FileNotFoundError(
            f"Data directory '{data_dir}' does not exist. "
            f"Create it and add at least one PDF file."
        )

    pdf_paths = sorted(data_dir.glob("*.pdf"))

    if not pdf_paths:
        raise FileNotFoundError(
            f"No PDF files found in '{data_dir}'. Add at least one .pdf file."
        )

    return pdf_paths


pdf_paths = discover_pdfs(DATA_DIR)
print(f"Discovered {len(pdf_paths)} PDF(s) in '{DATA_DIR}/':")
for path in pdf_paths:
    print(f"  - {path.name}")


Discovered 4 PDF(s) in 'data/':
  - artificial_intelligence_cheatsheet.pdf
  - deep_learning_cheatsheet.pdf
  - machine_learning_cheatsheet.pdf
  - python_cheatsheet.pdf


## Step 2 — Extract Text (`pypdf`)

<small>Returns one record per page of extractable text, tagged with page number and source file. Unreadable PDFs/pages are skipped, not fatal.</small>


In [121]:
def extract_pdf_pages(pdf_path: Path) -> List[Dict[str, Any]]:
    """Extract text per page as a list of {"text", "page", "source"} dicts."""
    pages: List[Dict[str, Any]] = []

    try:
        reader = PdfReader(str(pdf_path))
    except Exception as exc:
        print(f"  WARNING: could not open '{pdf_path.name}' — skipping. ({exc})")
        return pages

    for page_number, page in enumerate(reader.pages, start=1):
        try:
            page_text = page.extract_text() or ""
        except Exception as exc:
            print(f"  WARNING: failed to extract page {page_number} of "
                  f"'{pdf_path.name}' — skipping that page. ({exc})")
            continue

        page_text = page_text.strip()
        if page_text:
            pages.append({
                "text": page_text,
                "page": page_number,
                "source": pdf_path.name,
            })

    if not pages:
        print(f"  WARNING: no extractable text found in '{pdf_path.name}' "
              f"(it may be a scanned/image-only PDF).")

    return pages


## Step 3 — Chunk Text

<small>Slides a `CHUNK_SIZE`-character window forward with `CHUNK_OVERLAP` shared between consecutive chunks. Errors if overlap ≥ size.</small>


In [122]:
def chunk_text(text: str, chunk_size: int, chunk_overlap: int) -> List[str]:
    """Split text into overlapping, fixed-size chunks."""
    if chunk_overlap >= chunk_size:
        raise ValueError(
            f"chunk_overlap ({chunk_overlap}) must be smaller than "
            f"chunk_size ({chunk_size})."
        )

    chunks: List[str] = []
    start = 0
    text_length = len(text)

    while start < text_length:
        end = start + chunk_size
        chunk = text[start:end].strip()
        if chunk:
            chunks.append(chunk)
        if end >= text_length:
            break
        start = end - chunk_overlap  # advance, keeping overlap

    return chunks


## Step 4 — Build Chunk Records

<small>Combines Steps 2–3: extracts each PDF, chunks the text, and attaches source/page/id metadata for ChromaDB.</small>


In [123]:
def build_chunk_records(
    pdf_paths: List[Path],
    chunk_size: int,
    chunk_overlap: int,
) -> List[Dict[str, Any]]:
    """Extract + chunk every PDF, attaching id/source/page metadata."""
    records: List[Dict[str, Any]] = []

    for pdf_path in pdf_paths:
        print(f"Processing '{pdf_path.name}'...")
        pages = extract_pdf_pages(pdf_path)

        for page_info in pages:
            page_chunks = chunk_text(page_info["text"], chunk_size, chunk_overlap)

            for chunk_index, chunk in enumerate(page_chunks):
                records.append({
                    "id": f"{pdf_path.stem}_p{page_info['page']}_c{chunk_index}",
                    "text": chunk,
                    "source": page_info["source"],
                    "page": page_info["page"],
                    "chunk_index": chunk_index,
                })

    return records


chunk_records = build_chunk_records(pdf_paths, CHUNK_SIZE, CHUNK_OVERLAP)

print(f"\nBuilt {len(chunk_records)} chunks total.")
per_file_counts = {}
for record in chunk_records:
    per_file_counts[record["source"]] = per_file_counts.get(record["source"], 0) + 1
for source, count in per_file_counts.items():
    print(f"  - {source}: {count} chunks")


Processing 'artificial_intelligence_cheatsheet.pdf'...
Processing 'deep_learning_cheatsheet.pdf'...
Processing 'machine_learning_cheatsheet.pdf'...
Processing 'python_cheatsheet.pdf'...

Built 381 chunks total.
  - artificial_intelligence_cheatsheet.pdf: 82 chunks
  - deep_learning_cheatsheet.pdf: 54 chunks
  - machine_learning_cheatsheet.pdf: 70 chunks
  - python_cheatsheet.pdf: 175 chunks


## Step 5 — Generate Embeddings

<small>Loads `all-MiniLM-L6-v2` and encodes chunk texts into vectors for similarity search.</small>


In [124]:
def load_embedding_model(model_name: str) -> SentenceTransformer:
    """Load a sentence-transformers model; raises RuntimeError on failure."""
    try:
        return SentenceTransformer(model_name)
    except Exception as exc:
        raise RuntimeError(
            f"Failed to load embedding model '{model_name}'. Check your "
            f"internet connection and try again. Original error: {exc}"
        ) from exc


def embed_texts(model: SentenceTransformer, texts: List[str]) -> List[List[float]]:
    """Embed a list of texts, returning plain-list vectors."""
    try:
        embeddings = model.encode(
            texts,
            batch_size=32,
            show_progress_bar=True,
            convert_to_numpy=True,
        )
    except Exception as exc:
        raise RuntimeError(f"Embedding generation failed: {exc}") from exc

    return embeddings.tolist()


embedding_model = load_embedding_model(EMBEDDING_MODEL_NAME)

chunk_texts = [record["text"] for record in chunk_records]
t0 = time.time()
chunk_embeddings = embed_texts(embedding_model, chunk_texts)
print(f"Generated {len(chunk_embeddings)} embeddings "
      f"(dimension={len(chunk_embeddings[0])}) in {time.time() - t0:.2f}s")


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/12 [00:00<?, ?it/s]

Generated 381 embeddings (dimension=384) in 8.86s


## Step 6 — Store in ChromaDB

<small>Opens a persistent collection (cosine similarity) and upserts chunks + embeddings + metadata. Deterministic IDs avoid duplicates on re-run.</small>


In [125]:
def get_chroma_collection(persist_dir: str, collection_name: str):
    """Open or create a persistent ChromaDB collection (cosine similarity)."""
    try:
        chroma_client = chromadb.PersistentClient(path=persist_dir)
        collection = chroma_client.get_or_create_collection(
            name=collection_name,
            metadata={"hnsw:space": "cosine"},
        )
    except Exception as exc:
        raise RuntimeError(f"ChromaDB initialization failed: {exc}") from exc

    return collection


def index_chunks(collection, records: List[Dict[str, Any]], embeddings: List[List[float]]) -> None:
    """Upsert chunk text, embeddings, and metadata into ChromaDB."""
    try:
        collection.upsert(
            ids=[record["id"] for record in records],
            embeddings=embeddings,
            documents=[record["text"] for record in records],
            metadatas=[
                {
                    "source": record["source"],
                    "page": record["page"],
                    "chunk_index": record["chunk_index"],
                }
                for record in records
            ],
        )
    except Exception as exc:
        raise RuntimeError(f"Failed to store chunks in ChromaDB: {exc}") from exc


collection = get_chroma_collection(CHROMA_PERSIST_DIR, COLLECTION_NAME)
index_chunks(collection, chunk_records, chunk_embeddings)
print(f"ChromaDB collection '{COLLECTION_NAME}' now holds {collection.count()} chunks.")


ChromaDB collection 'week3_pdf_knowledge_base' now holds 381 chunks.


## Step 7 — Retrieve Top-K Chunks

<small>Embeds the question, queries ChromaDB, and converts distance to similarity (`1 - distance`).</small>


In [126]:
def retrieve_relevant_chunks(
    question: str,
    embedding_model: SentenceTransformer,
    collection,
    top_k: int,
) -> List[Dict[str, Any]]:
    """Return the top_k chunks most similar to the question."""
    try:
        question_embedding = embedding_model.encode(
            [question], convert_to_numpy=True
        ).tolist()
    except Exception as exc:
        raise RuntimeError(f"Failed to embed the question: {exc}") from exc

    try:
        results = collection.query(
            query_embeddings=question_embedding,
            n_results=top_k,
            include=["documents", "metadatas", "distances"],
        )
    except Exception as exc:
        raise RuntimeError(f"ChromaDB query failed: {exc}") from exc

    retrieved_chunks = []
    documents = results["documents"][0]
    metadatas = results["metadatas"][0]
    distances = results["distances"][0]

    for doc_text, metadata, distance in zip(documents, metadatas, distances):
        retrieved_chunks.append({
            "text": doc_text,
            "source": metadata["source"],
            "page": metadata["page"],
            "similarity": 1 - distance,
        })

    return retrieved_chunks


def display_retrieved_chunks(retrieved_chunks: List[Dict[str, Any]]) -> None:
    """Print each chunk's source, page, similarity, and a text preview."""
    if not retrieved_chunks:
        print("No relevant chunks were found in the knowledge base.")
        return

    print(f"Retrieved {len(retrieved_chunks)} relevant chunk(s):\n")
    for rank, chunk in enumerate(retrieved_chunks, start=1):
        preview = chunk["text"][:250] + ("..." if len(chunk["text"]) > 250 else "")
        print(f"[{rank}] source={chunk['source']}  page={chunk['page']}  "
              f"similarity={chunk['similarity']:.3f}")
        print(f"    {preview}\n")


## Step 8 — Generate a Grounded Answer

<small>Builds a prompt from the retrieved context + question, then calls the reused `OpenRouterClient.send_chat()`.</small>


In [127]:
RAG_SYSTEM_PROMPT = (
    "You are a helpful assistant that answers questions using ONLY the "
    "context provided below. Do not use any outside knowledge, and do not "
    "guess. If the answer cannot be found in the provided context, respond "
    "exactly with: 'The information is not available in the provided "
    "context.' Always mention which source/page the answer came from when "
    "you do find it in the context."
)


def build_rag_messages(question: str, retrieved_chunks: List[Dict[str, Any]]) -> List[Dict[str, str]]:
    """Build the chat messages for a retrieval-augmented question."""
    context_blocks = [
        f"[Source: {chunk['source']}, page {chunk['page']}]\n{chunk['text']}"
        for chunk in retrieved_chunks
    ]
    context_text = "\n\n---\n\n".join(context_blocks)

    user_message = (
        f"Context:\n{context_text}\n\n"
        f"Question: {question}\n\n"
        f"Answer the question using only the context above."
    )

    return [
        {"role": "system", "content": RAG_SYSTEM_PROMPT},
        {"role": "user", "content": user_message},
    ]


def generate_answer(
    client: "ChatApp.OpenRouterClient",
    model: str,
    question: str,
    retrieved_chunks: List[Dict[str, Any]],
) -> "tuple[str, Optional[float]]":
    """Ask the reused OpenRouter client to answer from retrieved context."""
    messages = build_rag_messages(question, retrieved_chunks)

    try:
        result, elapsed = client.send_chat(model, messages)
    except ChatApp.OpenRouterAPIError as exc:
        return f"Could not generate an answer: {exc}", None

    try:
        answer = result["choices"][0]["message"]["content"]
    except (KeyError, IndexError, TypeError):
        return "Received an unexpected response format from the API.", None

    return answer, elapsed


## Step 9 — Put It Together: `ask()`

<small>Single entry point: retrieves chunks, displays them, generates the answer, and prints it.</small>


In [128]:
def ask(question: str, top_k: int = TOP_K) -> str:
    """Answer a question end-to-end using the RAG pipeline."""
    print(f'Question: "{question}"\n')

    retrieved_chunks = retrieve_relevant_chunks(question, embedding_model, collection, top_k)
    display_retrieved_chunks(retrieved_chunks)

    if not retrieved_chunks:
        answer = "The information is not available in the provided context."
        print(f"=== Answer ===\n{answer}")
        return answer

    answer, elapsed = generate_answer(client, GENERATION_MODEL, question, retrieved_chunks)

    print("=== Answer ===")
    if elapsed is not None:
        print(f"(generated by {GENERATION_MODEL} in {elapsed:.2f}s)\n")
    print(answer)

    return answer


## Load API Key & Client (Reused)

<small>Calls `ChatApp.load_api_key()` and `ChatApp.OpenRouterClient(...)`, catching the missing-key case cleanly.</small>


In [129]:
try:
    api_key = ChatApp.load_api_key()
except SystemExit as exc:
    raise RuntimeError(
        "OPENROUTER_API_KEY is missing. Make sure a '.env' file is present "
        "in this same folder before continuing."
    ) from exc

client = ChatApp.OpenRouterClient(api_key)
print("OpenRouter client ready. Using model:", GENERATION_MODEL)


OpenRouter client ready. Using model: meta-llama/llama-3.1-8b-instruct


## Demo

<small>One answerable question, one out-of-scope question (to confirm the decline behavior).</small>


In [130]:
_ = ask("What is gradient descent and how is it used in machine learning?")

Question: "What is gradient descent and how is it used in machine learning?"

Retrieved 4 relevant chunk(s):

[1] source=machine_learning_cheatsheet.pdf  page=2  similarity=0.541
    ) + (1 − y) log(1 − z)
]
Linear regression Logistic regression SVM Neural Network
ÌCost function– The cost functionJ is commonly used to assess the performance of a model,
and is deﬁned with the loss functionL as follows:
J(θ) =
m∑
i=1
L(hθ(x(i)),y (...

[2] source=artificial_intelligence_cheatsheet.pdf  page=3  similarity=0.527
    Neural networks– Neural networks are a class of models that are built with layers. Com-
monly used types of neural networks include convolutional and recurrent neural networks. The
vocabulary around neural networks architectures is described in the ﬁ...

[3] source=deep_learning_cheatsheet.pdf  page=13  similarity=0.511
    h, let alone a normal-sized training set.
Ì Gradient checking – Gradient checking is a method used during the implementation of
the backward pass of a neura

In [131]:
_ = ask("What is the capital of France?")  # not covered by these PDFs — should decline

Question: "What is the capital of France?"

Retrieved 4 relevant chunk(s):

[1] source=python_cheatsheet.pdf  page=1  similarity=0.158
    stored as a string. 
Prompting for a value 
name = input("What's your name? ") 
print("Hello, " + name + "!") 
Prompting for numerical input 
age = input("How old are you? ") 
age = int(age) 
 
pi = input("What's the value of pi? ") 
pi = float(pi)

[2] source=deep_learning_cheatsheet.pdf  page=10  similarity=0.158
    t′> = 1
Remark: the attention scores are commonly used in image captioning and machine translation.
Ì Attention weight – The amount of attention that the outputy<t> should pay to the
activationa<t′> is given byα<t,t′> computed as follows:
α<t,t′> = e...

[3] source=python_cheatsheet.pdf  page=16  similarity=0.149
    le(self): 
        """Test names like David Lee Roth.""" 
        full_name = get_full_name('david', 
                'roth', 'lee') 
        self.assertEqual(full_name, 
                'David Lee Roth') 
 
unittest.main

## Try Your Own Question

<small>Run the cell below to type a question interactively.</small>


In [132]:
user_question = input("Enter your question: ").strip()
if user_question:
    _ = ask(user_question)
else:
    print("No question entered.")

Question: "What are types of machine learning?"

Retrieved 4 relevant chunk(s):

[1] source=machine_learning_cheatsheet.pdf  page=1  similarity=0.509
    CS 229 – Machine Learning https://stanford.edu/~shervine
Super VIP Cheatsheet: Machine Learning
Afshine Amidi and Shervine Amidi
October 6, 2018
Contents
1 Supervised Learning 2
1.1 Introduction to Supervised Learning. . . . . . . . . . . . . . . . ....

[2] source=machine_learning_cheatsheet.pdf  page=1  similarity=0.443
    t Vector Machines . . . . . . . . . . . . . . . . . . . . . . . . 3
1.5 Generative Learning . . . . . . . . . . . . . . . . . . . . . . . . . . . 4
1.5.1 Gaussian Discriminant Analysis . . . . . . . . . . . . . . . . . 4
1.5.2 Naive Bayes . . . . . ....

[3] source=artificial_intelligence_cheatsheet.pdf  page=3  similarity=0.416
    Neural networks– Neural networks are a class of models that are built with layers. Com-
monly used types of neural networks include convolutional and recurrent neural networks. The
vo

## Summary

<small>Pipeline: `discover_pdfs` → `extract_pdf_pages` → `chunk_text` → `build_chunk_records` → `embed_texts` → `index_chunks` → `retrieve_relevant_chunks` → `generate_answer` → `ask()`. Driven entirely by the config block and whatever PDFs are in `data/`. Tasks 2–5 not implemented — this is the foundation for them.</small>
